# Ettin full-width transformer exits — no Matryoshka training
Self-contained Colab A100 experiment. Compare **original pretrained Ettin-32M** with the **pre-prefix trained backbone**, initialize fresh
transformer exit heads, and optimize only full **384D** vectors. Full-depth predictions remain unchanged.
Both variants run sequentially with separate checkpoints and exports. The original MLM model is not already
a sentence embedding model; its full-depth retrieval score is an essential control.

All code is inline. Checkpoints, data identity, held-out comparisons, exports and final sanity reports go to
Drive. Run All resumes each variant's latest verified checkpoint. Definition cells can be edited before running the final orchestration cell. The full encoder and every selected exit are evaluated
at 384D, including early queries against a full-depth document index. This measures exit fidelity and retrieval
quality; it does not assume Ettin beats MiniLM/BGE. Use notebook 07 for the external baseline comparison.


In [ ]:
# Install into this notebook's kernel. Keep Colab's CUDA-enabled PyTorch.
# Restart the kernel after this cell if it already imported different versions.
%pip install -q transformers==5.1.0 datasets==4.5.0 accelerate==1.12.0 sentence-transformers==5.2.2 safetensors
%pip install -q scikit-learn==1.7.2 matplotlib


In [ ]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HOME'] = '/content/hf-cache' if os.path.isdir('/content') else '/tmp/encoder-hf-cache'
import copy, gc, hashlib, json, math, random, shutil, time, uuid
from pathlib import Path
from collections import defaultdict
from contextlib import nullcontext
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from transformers import AutoConfig, AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file, save_file
from tqdm.auto import tqdm
from google.colab import drive

drive.mount('/content/drive')
DEVICE = 'cuda'
assert torch.cuda.is_available(), 'Select an A100 GPU runtime in Colab.'
assert torch.cuda.is_bf16_supported(), 'This notebook uses BF16 on a supported GPU.'
print(torch.cuda.get_device_name(), 'PyTorch:', torch.__version__)
torch.set_float32_matmul_precision('high')
import csv, gzip, re
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, average_precision_score
from transformers.models.modernbert.modeling_modernbert import create_bidirectional_mask, create_bidirectional_sliding_window_mask
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer


## Configuration
Keep the settings unchanged to resume. Use a new `RUN_NAME` for a new source or budget.
Source requirements: notebook 06's `config.json`, `checkpoints/backbone/` (all retained generations), and
`data/mixture/`. No prefix or existing exit weights are loaded. Existing source data licensing/provenance
is retained: this remains a research continuation including MS MARCO and social-media data.

Default: 10,000 updates, batch 24, independent exits at depths 2/4/6/8. Each early path runs its backbone
prefix plus one extra transformer block. Set `EXIT_DEPTHS=list(range(1,10))` to train every intermediate exit.
For a quick smoke test choose a new name, `STEPS=2`, `EVAL_EVERY=1`, `VALIDATION_ROWS=8` and a smaller batch.


In [ ]:
RUN_NAME='ettin-fullwidth-exits-01'
VARIANTS=['pretrained','backbone_checkpoint']
SOURCE_RUN=Path('/content/drive/MyDrive/encoder_experiments/ettin-social-unified-02')
SOURCE_CHOICE='best'  # 'latest' or an integer backbone step, locked on the first run
ROOT=Path('/content/drive/MyDrive/encoder_experiments')/RUN_NAME
LOCAL=Path('/content/encoder-work')/RUN_NAME
EXIT_DEPTHS=[2,4,6,8]
STEPS=10000
BATCH_SIZE=24
MAX_LENGTH=256
LR=3e-5
EVAL_EVERY=250
SAVE_EVERY=100
VALIDATION_ROWS=128
SEED=45
RETRIEVAL_WEIGHT=.25
RELATIONAL_WEIGHT=.2
TEMPERATURE=.05
MIX_CYCLE=['msmarco','go_emotions','nq','tweet_topic','msmarco','all_nli','squad','go_emotions','tweet_topic']
ETTIN=dict(id='jhu-clsp/ettin-encoder-32m',revision='1b8ba06455dd44f80fc9c1ca9e22806157a57379')
NANO=dict(id='sentence-transformers/NanoBEIR-en',revision='beb106fbcfaa599c508c667041bf8c85fd78736b')
BENCHMARK_NAMES=['NanoNQ','NanoHotpotQA','NanoMSMARCO']
assert ROOT.resolve()!=SOURCE_RUN.resolve()
ROOT.mkdir(parents=True,exist_ok=True);LOCAL.mkdir(parents=True,exist_ok=True)
EXPERIMENT_ROOT=ROOT;EXPERIMENT_LOCAL=LOCAL
assert STEPS>0 and BATCH_SIZE>1 and EVAL_EVERY>0 and SAVE_EVERY>0
CFG=dict(short_length=MAX_LENGTH,teacher_batch_size=16,eval_batch_size=32)
def digest(x):return hashlib.sha256(json.dumps(x,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
def file_sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''):h.update(chunk)
    return h.hexdigest()
def write_json(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp');tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False));os.replace(tmp,path)
def seed_everything(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
def mean_pool(h,mask):
    weights=mask.unsqueeze(-1).float();return (h.float()*weights).sum(1)/weights.sum(1).clamp_min(1)
def backbone_digest():
    h=hashlib.sha256()
    for k,v in backbone.state_dict().items():
        h.update(k.encode());h.update(v.detach().cpu().contiguous().view(torch.uint8).numpy().tobytes())
    return h.hexdigest()
seed_everything(SEED)


## Verify source and load the frozen backbone


In [ ]:
def load_variant():
    global BACKBONE_SHA, DEPTHS, DIMS, NUM_LAYERS, SOURCE_MODEL_SPEC, WIDTH, backbone, choices, info, lock, manifest, parent, read_source, selected, source_contract, source_id, source_info, spec, state, tokenizer, typ, wanted, weights
    source_contract=json.loads((SOURCE_RUN/'config.json').read_text())
    source_id=digest(source_contract)
    assert source_contract['ettin']==ETTIN

    def read_source(path):
        manifest=json.loads((path/'COMMITTED.json').read_text())
        assert manifest['contract']==source_id and file_sha(path/'state.pt')==manifest['sha256']
        state=torch.load(path/'state.pt',map_location='cpu',weights_only=False)  # your own checkpoint only
        assert state['contract']==source_id and state['step']==manifest['step']
        assert state['stage']=='backbone' and state['family']=='ettin'
        return state,manifest

    if SOURCE_MODE=='backbone_checkpoint':
        parent=SOURCE_RUN/'checkpoints'/'backbone'
        choices=sorted((p for p in parent.glob('*') if not p.name.startswith('.') and (p/'COMMITTED.json').is_file()),reverse=True)
        if not choices:raise FileNotFoundError(f'No committed backbone generations in {parent}. Copy them or restore them on Drive.')
        lock=ROOT/'source.json'
        if lock.exists():
            info=json.loads(lock.read_text());assert info['mode']==SOURCE_MODE and info['choice']==SOURCE_CHOICE and info['run']==str(SOURCE_RUN)
            wanted=info['step']
        else:
            state,manifest=read_source(choices[0])
            wanted=state['best_step'] if SOURCE_CHOICE=='best' else state['step'] if SOURCE_CHOICE=='latest' else int(SOURCE_CHOICE)
            del state
        selected=next((p for p in choices if p.name.startswith(f'{wanted:08d}-')),None)
        if selected is None:raise FileNotFoundError(f'Required pre-prefix backbone step {wanted} is missing.')
        state,manifest=read_source(selected)
        source_info=dict(mode=SOURCE_MODE,choice=SOURCE_CHOICE,run=str(SOURCE_RUN),step=wanted,sha=manifest['sha256'])
        if lock.exists():assert info==source_info,'Locked source changed.'
        else:write_json(lock,source_info)
        spec=copy.deepcopy(state['spec']);typ=spec.pop('model_type');assert typ=='modernbert'
        backbone=AutoModel.from_config(AutoConfig.for_model(typ,**spec),attn_implementation='sdpa')
        weights={k.removeprefix('backbone.'):v for k,v in state['model'].items() if k.startswith('backbone.')}
        backbone.load_state_dict(weights,strict=True);del state,weights
    else:
        assert SOURCE_MODE=='pretrained'
        source_info=dict(mode=SOURCE_MODE,model=ETTIN)
        backbone=AutoModel.from_pretrained(ETTIN['id'],revision=ETTIN['revision'],attn_implementation='sdpa')
    tokenizer=AutoTokenizer.from_pretrained(ETTIN['id'],revision=ETTIN['revision'])
    backbone.to(DEVICE).eval().requires_grad_(False);backbone.gradient_checkpointing_disable()
    NUM_LAYERS=backbone.config.num_hidden_layers;WIDTH=backbone.config.hidden_size
    assert WIDTH==384 and MAX_LENGTH<=backbone.config.max_position_embeddings
    DEPTHS=list(EXIT_DEPTHS);DIMS=[WIDTH]
    assert DEPTHS==sorted(set(DEPTHS)) and DEPTHS and all(1<=d<NUM_LAYERS for d in DEPTHS)
    BACKBONE_SHA=backbone_digest()
    SOURCE_MODEL_SPEC=dict(kind='transformer',depths=DEPTHS,width=WIDTH,backbone_sha=BACKBONE_SHA,independent_heads=True)
    print('Frozen source:',source_info,'dimensions:',DIMS,'depths:',DEPTHS)


In [ ]:
def tokenize(texts):
    return tokenizer(texts,padding=True,truncation=True,max_length=CFG['short_length'],return_tensors='pt').to(DEVICE)

@torch.no_grad()
def frozen_features(tokens,depths):
    backbone.eval()
    last=max(depths);assert 1<=last<=NUM_LAYERS
    h=backbone.embeddings(input_ids=tokens['input_ids'])
    positions=torch.arange(h.shape[1],device=h.device).unsqueeze(0)
    mask_args=dict(config=backbone.config,input_embeds=h,attention_mask=tokens['attention_mask'])
    masks={'full_attention':create_bidirectional_mask(**mask_args),
           'sliding_attention':create_bidirectional_sliding_window_mask(**mask_args)}
    rope={kind:backbone.rotary_emb(h,positions,kind) for kind in set(backbone.config.layer_types)}
    outputs={}
    for depth,layer in enumerate(backbone.layers[:last],1):
        h=layer(h,attention_mask=masks[layer.attention_type],position_embeddings=rope[layer.attention_type])
        if depth in depths:outputs[depth]=h.detach()
    return outputs,dict(masks=masks,rope=rope)

def native_pool(hidden,mask):
    return mean_pool(backbone.final_norm(hidden),mask)


class ExitHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.norm=copy.deepcopy(backbone.final_norm).requires_grad_(False)
        self.block=copy.deepcopy(backbone.layers[-1]).requires_grad_(True)
        self.block.gradient_checkpointing=False
    def forward(self,h,mask,context):
        typ=self.block.attention_type
        h=self.block(h,attention_mask=context['masks'][typ],position_embeddings=context['rope'][typ])
        return mean_pool(self.norm(h),mask)

class ExitBank(nn.Module):
    def __init__(self):
        super().__init__();self.family='ettin-exits';self.spec=copy.deepcopy(SOURCE_MODEL_SPEC)
        self.heads=nn.ModuleDict({str(d):ExitHead() for d in DEPTHS})
    def forward(self,depth,h,mask,context):return self.heads[str(depth)](h,mask,context)

def early_raw(tokens,depth,bank=None):
    features,context=frozen_features(tokens,[depth])
    if depth==NUM_LAYERS or bank is None:return native_pool(features[depth],tokens['attention_mask'])
    return bank(depth,features[depth],tokens['attention_mask'],context)

@torch.inference_mode()
def encode_profiles(texts,bank=None,batch_size=None):
    if bank is not None:bank.eval()
    result={d:[] for d in DEPTHS+[NUM_LAYERS]}
    for start in range(0,len(texts),batch_size or CFG['teacher_batch_size']):
        tokens=tokenize(texts[start:start+(batch_size or CFG['teacher_batch_size'])])
        with torch.autocast('cuda',dtype=torch.bfloat16):
            features,context=frozen_features(tokens,DEPTHS+[NUM_LAYERS])
            for depth in result:
                z=native_pool(features[depth],tokens['attention_mask']) if bank is None or depth==NUM_LAYERS else bank(depth,features[depth],tokens['attention_mask'],context)
                z=z.float().cpu()
                assert torch.isfinite(z).all() and (z.norm(dim=-1)>1e-8).all(), 'Invalid embeddings'
                result[depth].append(z)
    return {depth:torch.cat(parts) for depth,parts in result.items()}


In [ ]:
def check_execution():
    global canonical, counts, hook, hooks, probe, tokens
    probe=['Recover a forgotten password.','Books can be borrowed at the library.']
    tokens=tokenize(probe)
    with torch.inference_mode(),torch.autocast('cuda',dtype=torch.bfloat16):
        canonical=mean_pool(backbone(**tokens).last_hidden_state,tokens['attention_mask'])
        torch.testing.assert_close(canonical,early_raw(tokens,NUM_LAYERS),atol=1e-5,rtol=1e-4)
    counts=[0]*NUM_LAYERS
    hooks=[layer.register_forward_hook(lambda module,args,output,i=i:counts.__setitem__(i,counts[i]+1)) for i,layer in enumerate(backbone.layers)]
    try:
        with torch.inference_mode(),torch.autocast('cuda',dtype=torch.bfloat16):early_raw(tokens,min(DEPTHS))
    finally:
        for hook in hooks:hook.remove()
    assert counts==[int(i<min(DEPTHS)) for i in range(NUM_LAYERS)]
    print('Canonical full-depth parity and actual early-exit execution passed.')


## Reuse the verified training mixture and held-out splits
GoEmotions and TweetTopic contribute text for teacher alignment; auxiliary classifiers are not loaded.
Retrieval batches additionally use labeled positive/negative pairs. Exact benchmark/sanity text overlaps
are filtered here; the upstream backbone may have other pretraining overlap. Test splits never select heads.


In [ ]:
# A pinned, labeled retrieval benchmark. Never used for optimization or early stopping.
BENCHMARKS = {}
def load_benchmarks():
    if BENCHMARKS:
        return BENCHMARKS
    for name in BENCHMARK_NAMES:
        parts = {part: load_dataset('parquet', data_files={'train':f"hf://datasets/{NANO['id']}@{NANO['revision']}/{part}/{name}-*.parquet"}, split='train')
                 for part in ('queries', 'corpus', 'qrels')}
        queries = {str(r['_id']): r['text'] for r in parts['queries'] if r['text'].strip()}
        corpus = {str(r['_id']): r['text'] for r in parts['corpus'] if r['text'].strip()}
        relevant = defaultdict(set)
        for r in parts['qrels']:
            ids = r['corpus-id'] if isinstance(r['corpus-id'], list) else [r['corpus-id']]
            relevant[str(r['query-id'])].update(str(x) for x in ids)
        BENCHMARKS[name] = dict(queries=queries, corpus=corpus, relevant=dict(relevant))
    return BENCHMARKS

def retrieval_metrics(q, p, relevant, query_ids, corpus_ids):
    scores = q @ p.T
    ranking = scores.argsort(dim=1, descending=True)
    sums = defaultdict(float)
    for i, qid in enumerate(query_ids):
        truth = relevant[qid]
        top = [corpus_ids[j] for j in ranking[i, :10].tolist()]
        hits = [int(x in truth) for x in top]
        sums['hit@1'] += hits[0]
        sums['hit@5'] += bool(sum(hits[:5]))
        sums['recall@10'] += sum(hits) / len(truth)
        sums['mrr@10'] += next((1/(j+1) for j,h in enumerate(hits) if h), 0)
        dcg = sum(h/math.log2(j+2) for j,h in enumerate(hits))
        ideal = sum(1/math.log2(j+2) for j in range(min(len(truth), 10)))
        sums['ndcg@10'] += dcg / ideal
    return {k: v/len(query_ids) for k,v in sums.items()}


In [ ]:
SANITY_CASES = [
    ('I forgot my account password.', 'Use the password reset link to regain access to your account.',
     'The tomato plants need watering every morning.'),
    ('How can I get my money back for this purchase?', 'Request a refund for your order through customer support.',
     'The train leaves the station at six in the evening.'),
    ('My phone battery runs out too quickly.', 'Reduce screen brightness and background apps to extend battery life.',
     'The museum has opened a new exhibition of ancient pottery.'),
    ('How do I bake a loaf of bread?', 'Mix flour, water and yeast, let the dough rise, then bake it in the oven.',
     'The football team won the championship yesterday.'),
    ('I want to cancel my monthly subscription.', 'Open your billing settings and choose cancel subscription.',
     'Migrating birds travel south during the winter.'),
    ('My laptop cannot connect to Wi-Fi.', 'Check the wireless network settings and restart your router.',
     'The orchestra performed a symphony at the concert hall.'),
    ('How can I track the package I ordered?', 'Enter your parcel tracking number to see its delivery status.',
     'Stretching your legs before a run can help improve flexibility.'),
    ('How do I change the language in the app?', 'Choose your preferred display language in the application settings.',
     'The recipe calls for two eggs and a cup of milk.'),
    ('I need to book a visit with a doctor.', 'Contact the clinic to schedule a medical appointment.',
     'The new telescope can observe distant galaxies.'),
    ('Where can I find books to borrow?', 'Visit the public library to check out books with your library card.',
     'The car needs new tires before the road trip.'),
    ('How can I recover a file that I deleted?', 'Look in the recycle bin or restore the document from a backup.',
     'The beach is covered in shells after the tide goes out.'),
    ('I want to learn to play the guitar.', 'Practice basic chords and strumming patterns on your guitar each day.',
     'The city council approved construction of a new bridge.'),
]

def sanity_scores(raw, cases, dimensions):
    # Pure scoring function: no model, tokenizer or teacher dependency.
    raw = raw.detach().float().cpu()
    n = len(cases)
    assert raw.ndim == 2 and len(raw) == 3*n
    finite = bool(torch.isfinite(raw).all())
    norms = raw.norm(dim=-1)
    nonzero = finite and bool((norms > 1e-8).all())
    result = dict(finite=finite, nonzero=nonzero,
                  raw_norm_min=float(norms.min()) if finite else None,
                  raw_norm_mean=float(norms.mean()) if finite else None,
                  profiles={})
    if not nonzero:
        return result
    for dim in dimensions:
        prefix_norms = raw[:, :dim].norm(dim=-1)
        z = F.normalize(raw[:, :dim], dim=-1)
        q, positive, unrelated = z.split(n)
        pos = (q*positive).sum(-1); neg = (q*unrelated).sum(-1)
        scores = q @ torch.cat((positive, unrelated)).T
        positive_mask = torch.zeros_like(scores, dtype=torch.bool)
        positive_mask[torch.arange(n), torch.arange(n)] = True
        # Conservative tie handling: a collapsed model must not pass by index order.
        ranks = 1 + ((scores >= pos[:,None]-1e-6) & ~positive_mask).sum(-1)
        pairwise = z @ z.T
        offdiag = pairwise[~torch.eye(len(z),dtype=torch.bool)]
        result['profiles'][str(dim)] = dict(
            nonzero_prefix=bool((prefix_norms > 1e-8).all()),
            easy_accuracy=float((pos > neg+1e-6).float().mean()),
            mean_margin=float((pos-neg).mean()),
            hit_at_1=float((ranks <= 1).float().mean()),
            hit_at_5=float((ranks <= 5).float().mean()),
            mrr=float((1/ranks.float()).mean()),
            normalized_spread=float((z-z.mean(0)).square().sum(-1).mean()),
            off_diagonal_cosine_mean=float(offdiag.mean()),
            examples=[dict(query=cases[i][0], positive=cases[i][1], unrelated=cases[i][2],
                           positive_cosine=float(pos[i]), unrelated_cosine=float(neg[i]),
                           positive_rank=int(ranks[i]), easy_correct=bool(pos[i]>neg[i]+1e-6))
                      for i in range(n)])
    return result


In [ ]:
def prepare_data():
    global CONTRACT_ID, DATA_ID, FROZEN, b, contract, corpus, excluded, fm, meta, mixture, name, parts, row_texts, rows, seen, split, text_id, tmp
    mixture=SOURCE_RUN/'data'/'mixture';meta=json.loads((mixture/'COMMITTED.json').read_text())
    assert meta['contract']==source_id and file_sha(mixture/'pairs.json')==meta['sha256']
    corpus=json.loads((mixture/'pairs.json').read_text())
    def text_id(s):return hashlib.sha256(' '.join(s.split()).casefold().encode()).hexdigest()
    def row_texts(r):return [r[k] for k in ('query','positive','negative','text','sentence1','sentence2') if k in r]
    excluded={text_id(t) for case in SANITY_CASES for t in case}
    for b in load_benchmarks().values():excluded.update(text_id(t) for t in list(b['queries'].values())+list(b['corpus'].values()))
    for name,parts in corpus.items():
        for split,rows in parts.items():parts[split]=[r for r in rows if not any(text_id(t) in excluded for t in row_texts(r))]
    seen={s:set() for s in ('train','validation','test')}
    for parts in corpus.values():
        for split,rows in parts.items():seen[split].update(text_id(t) for r in rows for t in row_texts(r))
    assert not (seen['train']&seen['validation'] or seen['train']&seen['test'] or seen['validation']&seen['test'])
    assert set(MIX_CYCLE)<=set(corpus)
    assert all(len(corpus[n]['train'])>=BATCH_SIZE and len(corpus[n]['validation'])>=2 for n in set(MIX_CYCLE))
    DATA_ID=digest(corpus)
    contract=dict(source=source_info,backbone_sha=BACKBONE_SHA,data_id=DATA_ID,steps=STEPS,batch=BATCH_SIZE,
                  max_length=MAX_LENGTH,lr=LR,seed=SEED,depths=DEPTHS,dimensions=DIMS,mix=MIX_CYCLE,
                  eval_every=EVAL_EVERY,save_every=SAVE_EVERY,validation_rows=VALIDATION_ROWS,
                  retrieval_weight=RETRIEVAL_WEIGHT,relational_weight=RELATIONAL_WEIGHT,temperature=TEMPERATURE,
                  nano=NANO,implementation=1)
    CONTRACT_ID=digest(contract)
    if (ROOT/'config.json').exists():assert json.loads((ROOT/'config.json').read_text())==contract,'Use a new run name for changed settings.'
    else:write_json(ROOT/'config.json',contract)
    write_json(ROOT/'LICENSE_PROVENANCE.json',dict(source_run=str(SOURCE_RUN),licenses=source_contract.get('licenses'),
        note='Research continuation; data terms from notebook 06 remain applicable. No new blanket license on weights.'))
    # A verified snapshot lets checkpoints remain usable even if the old run is later removed.
    FROZEN=ROOT/'frozen-backbone'
    if not FROZEN.exists():
        tmp=ROOT/('.uploading-backbone-'+uuid.uuid4().hex);tmp.mkdir()
        backbone.save_pretrained(tmp/'backbone',safe_serialization=True);tokenizer.save_pretrained(tmp/'tokenizer')
        write_json(tmp/'COMMITTED.json',dict(contract=CONTRACT_ID,checksums={str(p.relative_to(tmp)):file_sha(p) for p in tmp.rglob('*') if p.is_file()}))
        os.replace(tmp,FROZEN)
    fm=json.loads((FROZEN/'COMMITTED.json').read_text());assert fm['contract']==CONTRACT_ID
    assert all(file_sha(FROZEN/name)==sha for name,sha in fm['checksums'].items())
    print({n:{s:len(r) for s,r in p.items()} for n,p in corpus.items()})


## Full-vector loss and validation
Each head receives the same batches and update count. Optimize cosine alignment plus pairwise-similarity
alignment to the frozen full-depth teacher. Retrieval batches add supervised ranking, masking duplicate and
teacher-suspected false negatives. Every term uses all 384 coordinates; there are no dimension-prefix losses.
Validation reports teacher cosine and same-exit/full-document MRR separately. Selection uses mean exit
teacher cosine across sources (fidelity); NanoBEIR is reporting-only. The unchanged full-depth teacher provides
a reference, not a promise of embedding quality.


In [ ]:
class RowSampler:
    def __init__(self,n,seed):
        assert n>0
        self.n=n;self.generator=torch.Generator().manual_seed(seed)
        self.order=torch.randperm(n,generator=self.generator);self.cursor=0;self.epoch=0;self.drawn=0
    def next(self,batch):
        assert batch<=self.n
        ids=[]
        while len(ids)<batch:
            if self.cursor==self.n:
                self.order=torch.randperm(self.n,generator=self.generator);self.cursor=0;self.epoch+=1
            size=min(batch-len(ids),self.n-self.cursor)
            ids.extend(self.order[self.cursor:self.cursor+size].tolist());self.cursor+=size
        self.drawn+=batch;return ids
    def state_dict(self):
        return dict(n=self.n,order=self.order,cursor=self.cursor,epoch=self.epoch,drawn=self.drawn,generator=self.generator.get_state())
    def load_state_dict(self,state):
        assert state['n']==self.n and 0<=state['cursor']<=self.n
        self.order=state['order'].clone();self.cursor=state['cursor'];self.epoch=state['epoch'];self.drawn=state['drawn'];self.generator.set_state(state['generator'])


In [ ]:
class Sampler:
    def __init__(self):
        self.turn=0;self.samplers={n:RowSampler(len(corpus[n]['train']),SEED+i) for i,n in enumerate(sorted(set(MIX_CYCLE)))}
    def next(self):
        name=MIX_CYCLE[self.turn%len(MIX_CYCLE)];self.turn+=1
        return name,[corpus[name]['train'][i] for i in self.samplers[name].next(BATCH_SIZE)]
    def state_dict(self):return dict(turn=self.turn,samplers={n:s.state_dict() for n,s in self.samplers.items()})
    def load_state_dict(self,state):
        self.turn=state['turn']
        for n,s in state['samplers'].items():self.samplers[n].load_state_dict(s)
def texts_for(rows):
    if 'query' in rows[0]:
        keys=['query','positive']+(['negative'] if all('negative' in r for r in rows) else [])
        return [r[k] for k in keys for r in rows]
    return [t for r in rows for t in row_texts(r)]
def loss_backward(rows):
    tokens=tokenize(texts_for(rows))
    with torch.no_grad(),torch.autocast('cuda',dtype=torch.bfloat16):
        features,ctx=frozen_features(tokens,DEPTHS+[NUM_LAYERS])
        target=F.normalize(native_pool(features[NUM_LAYERS],tokens['attention_mask']).float(),dim=-1)
    loss_value=0.
    for depth in DEPTHS:
        with torch.autocast('cuda',dtype=torch.bfloat16):raw=bank(depth,features[depth],tokens['attention_mask'],ctx)
        z=F.normalize(raw.float(),dim=-1)
        loss=(1-(z*target).sum(-1)).mean()+RELATIONAL_WEIGHT*F.mse_loss(z@z.T,target@target.T)
        if 'query' in rows[0]:
            n=len(rows);teacher_scores=target[:n]@target[n:].T
            positive=teacher_scores[torch.arange(n),torch.arange(n)]
            keep=teacher_scores < positive[:,None]-.05
            keys=['positive']+(['negative'] if all('negative' in r for r in rows) else [])
            docs=[r[k] for k in keys for r in rows]
            for i,r in enumerate(rows):
                for j,doc in enumerate(docs):
                    if text_id(doc)==text_id(r['positive']):keep[i,j]=False
            keep[torch.arange(n),torch.arange(n)]=True
            logits=(z[:n]@z[n:].T/TEMPERATURE).masked_fill(~keep,-1e9)
            loss+=RETRIEVAL_WEIGHT*F.cross_entropy(logits,torch.arange(n,device=DEVICE))
        loss=loss/len(DEPTHS);assert torch.isfinite(loss)
        loss.backward();loss_value+=float(loss.detach())
    return loss_value
@torch.inference_mode()
def validate(split='validation'):
    bank.eval();records=[]
    for name in sorted(set(MIX_CYCLE)):
        rows=corpus[name][split][:VALIDATION_ROWS]
        vectors=encode_profiles(texts_for(rows),bank)
        z={d:F.normalize(v,dim=-1) for d,v in vectors.items()};full=z[NUM_LAYERS]
        for depth in DEPTHS+[NUM_LAYERS]:
            rec=dict(source=name,depth=depth,cosine=float((z[depth]*full).sum(-1).mean()))
            if 'query' in rows[0]:
                count=len(rows);ids=[text_id(r['positive']) for r in rows]
                relevant=torch.tensor([[a==b for b in ids] for a in ids])
                for mode,docs in [('same_exit',z[depth]),('full_corpus',full)]:
                    scores=z[depth][:count]@docs[count:2*count].T
                    order=scores.argsort(-1,descending=True)
                    ranks=relevant.gather(1,order).float().argmax(-1)+1
                    rec[mode+'_mrr']=float((1/ranks.float()).mean())
            records.append(rec)
    return dict(score=float(np.mean([r['cosine'] for r in records if r['depth']!=NUM_LAYERS])),profiles=records)


## Restartable training and verified checkpoints


In [ ]:
# Completed generations are immutable. Upload to a temporary directory; COMMITTED.json is written last.
# Resume discovers the newest verified generation, rather than trusting a possibly stale "latest" pointer.
def file_sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''): h.update(chunk)
    return h.hexdigest()

def rng_state():
    return dict(python=random.getstate(), numpy=np.random.get_state(), cpu=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all())
def restore_rng(state):
    random.setstate(state['python']); np.random.set_state(state['numpy'])
    torch.set_rng_state(state['cpu'])
    if torch.cuda.is_available(): torch.cuda.set_rng_state_all(state['cuda'])

def save_checkpoint(stage, step, model, optimizer, scheduler, sampler, best_metric, best_step, total_steps):
    parent=ROOT/'checkpoints'/stage; parent.mkdir(parents=True,exist_ok=True)
    generation=f'{step:08d}-{uuid.uuid4().hex[:8]}'
    local=LOCAL/(generation+'.pt')
    torch.save(dict(stage=stage,step=step,total_steps=total_steps,contract=CONTRACT_ID,data_id=DATA_ID,
                    family=model.family,spec=model.spec,model=model.state_dict(),optimizer=optimizer.state_dict(),
                    scheduler=scheduler.state_dict(),sampler=sampler.state_dict(),rng=rng_state(),
                    best_metric=best_metric,best_step=best_step),local)
    tmp=parent/('.uploading-'+generation); tmp.mkdir()
    shutil.copyfile(local,tmp/'state.pt')
    sha=file_sha(local)
    assert file_sha(tmp/'state.pt') == sha, 'Drive checkpoint copy failed checksum verification'
    write_json(tmp/'COMMITTED.json',dict(step=step,sha256=sha,contract=CONTRACT_ID))
    os.replace(tmp,parent/generation)
    local.unlink()
    # Retain latest two plus the generation containing the best validation result.
    generations=sorted(p for p in parent.iterdir() if (p/'COMMITTED.json').exists() and not p.name.startswith('.'))
    keep={p.name for p in generations[-2:]}
    keep.update(p.name for p in generations if int(p.name.split('-')[0]) == best_step)
    for p in generations:
        if p.name not in keep: shutil.rmtree(p)
    return parent/generation

def latest_checkpoint(stage):
    parent=ROOT/'checkpoints'/stage
    if not parent.exists(): return None
    candidates=sorted((p for p in parent.iterdir() if not p.name.startswith('.') and (p/'COMMITTED.json').exists()),reverse=True)
    for path in candidates:
        try:
            manifest=json.loads((path/'COMMITTED.json').read_text())
            assert manifest['contract']==CONTRACT_ID
            assert file_sha(path/'state.pt')==manifest['sha256']
            return path
        except (OSError,ValueError,AssertionError) as e:
            print('Skipping invalid checkpoint',path,e)
    if candidates: raise RuntimeError('No valid checkpoint remains; inspect Drive files.')
    return None

def load_checkpoint(stage):
    path=latest_checkpoint(stage)
    if path is None: return None
    local=LOCAL/(stage+'-resume.pt'); shutil.copyfile(path/'state.pt',local)
    # Only our own checkpoints, never arbitrary downloaded pickle files.
    assert file_sha(local)==json.loads((path/'COMMITTED.json').read_text())['sha256']
    state=torch.load(local,map_location='cpu',weights_only=False)
    assert state['contract']==CONTRACT_ID and state['data_id']==DATA_ID
    return state


In [ ]:
def restore_best(model,stage):
    state=load_checkpoint(stage)
    assert state is not None
    best=state['best_step']; del state
    parent=ROOT/'checkpoints'/stage
    choices=sorted(p for p in parent.iterdir() if p.name.startswith(f'{best:08d}-') and (p/'COMMITTED.json').exists())
    assert choices, 'Best checkpoint is missing'
    path=choices[-1];meta=json.loads((path/'COMMITTED.json').read_text())
    assert file_sha(path/'state.pt')==meta['sha256']
    state=torch.load(path/'state.pt',map_location='cpu',weights_only=False)
    model.load_state_dict(state['model'],strict=True)
    print('Selected',stage,'validation-best step',best)
    return model


In [ ]:
def train_heads():
    global SELECTED, bank, best, best_step, f, improved, initial, loss, metrics, name, optimizer, rec, rows, sampler, scheduler, start, state, step
    # Each run keeps latest two generations plus the validation-best generation.
    # Frozen backbone snapshot is shared by all exit checkpoints.
    seed_everything(SEED)
    bank=ExitBank().to(DEVICE)
    optimizer=torch.optim.AdamW([p for p in bank.parameters() if p.requires_grad],lr=LR,weight_decay=.01)
    scheduler=get_cosine_schedule_with_warmup(optimizer,num_warmup_steps=min(100,max(1,STEPS//10)),num_training_steps=STEPS)
    sampler=Sampler()
    state=load_checkpoint('exits')
    if state is None:
        initial=validate();best=initial['score'];best_step=0;start=0
        write_json(ROOT/'evaluations'/'initial-validation.json',initial)
        save_checkpoint('exits',0,bank,optimizer,scheduler,sampler,best,best_step,STEPS)
    else:
        assert state['spec']==bank.spec and state['total_steps']==STEPS
        bank.load_state_dict(state['model']);optimizer.load_state_dict(state['optimizer']);scheduler.load_state_dict(state['scheduler'])
        sampler.load_state_dict(state['sampler']);restore_rng(state['rng'])
        start=state['step'];best=state['best_metric'];best_step=state['best_step'];del state
    print('Starting/resuming',start,'/',STEPS)
    for step in range(start+1,STEPS+1):
        name,rows=sampler.next();bank.train();optimizer.zero_grad(set_to_none=True)
        loss=loss_backward(rows)
        nn.utils.clip_grad_norm_([p for p in bank.parameters() if p.requires_grad],1.,error_if_nonfinite=True)
        optimizer.step();scheduler.step()
        rec=dict(step=step,source=name,loss=loss);improved=False
        if step==1 or step%25==0:print('exits',step,'/',STEPS,name,'loss',round(loss,5),flush=True)
        if step%EVAL_EVERY==0 or step==STEPS:
            metrics=validate();rec['validation']=metrics
            if metrics['score']>best:best=metrics['score'];best_step=step;improved=True
            print('validation cosine',round(metrics['score'],5),'best step',best_step,flush=True)
        with (ROOT/'metrics.jsonl').open('a') as f:f.write(json.dumps(rec)+'\n')
        if improved or step%SAVE_EVERY==0 or step==STEPS:
            save_checkpoint('exits',step,bank,optimizer,scheduler,sampler,best,best_step,STEPS)
    bank=restore_best(bank,'exits').eval()
    SELECTED=dict(step=best_step,score=best,completed_steps=STEPS)
    write_json(ROOT/'selected.json',SELECTED)
    write_json(ROOT/'evaluations'/'held-out-test.json',validate('test'))
    assert backbone_digest()==BACKBONE_SHA,'Frozen backbone changed.'
    print('Selected',SELECTED)


## Independent retrieval: full width at each depth


In [ ]:
def evaluate_retrieval():
    global RESULTS, b, cids, depth, docdepth, metrics, mode, name, p, q, qids, rec, rows
    RESULTS=[]
    for name,b in load_benchmarks().items():
        cids=list(b['corpus']);qids=[q for q in b['queries'] if b['relevant'].get(q)]
        assert all(b['relevant'][q]<=set(cids) for q in qids)
        q=encode_profiles([b['queries'][k] for k in qids],bank)
        p=encode_profiles(list(b['corpus'].values()),bank)
        for depth in DEPTHS+[NUM_LAYERS]:
            for mode,docdepth in [('same_exit',depth),('full_corpus',NUM_LAYERS)]:
                metrics=retrieval_metrics(F.normalize(q[depth],dim=-1),F.normalize(p[docdepth],dim=-1),b['relevant'],qids,cids)
                rec=dict(dataset=name,depth=depth,dim=WIDTH,mode=mode,**metrics);RESULTS.append(rec);print(rec,flush=True)
    write_json(ROOT/'evaluations'/'retrieval.json',RESULTS)
    for name in BENCHMARK_NAMES:
        for mode in ['same_exit','full_corpus']:
            rows=[r for r in RESULTS if r['dataset']==name and r['mode']==mode]
            plt.plot([r['depth'] for r in rows],[r['ndcg@10'] for r in rows],marker='o',label=mode)
        plt.title(name+' — 384D');plt.xlabel('Backbone depth (early exits add one block)');plt.ylabel('NDCG@10');plt.legend();plt.show()


## Export for notebook 07
The export has the same inference layout as notebook 06; set notebook 07 to `ARTIFACT_MODE='export'`
and point it to this run's `exports/social-encoder`. It will evaluate only the trained 384D dimension.
No Matryoshka or old exit-head weights are loaded. Reload parity is checked below.


In [ ]:
def export_variant():
    global EXPORT, after, backbone, before, d, loaded, manifest, original_backbone, original_tokenizer, probe, tmp, tokenizer
    EXPORT=ROOT/'exports'/'social-encoder'
    if not EXPORT.exists():
        tmp=ROOT/('export-tmp-'+uuid.uuid4().hex);tmp.mkdir()
        backbone.save_pretrained(tmp/'backbone',safe_serialization=True);tokenizer.save_pretrained(tmp/'tokenizer')
        save_file({k:v.detach().cpu().contiguous() for k,v in bank.state_dict().items()},str(tmp/'exits.safetensors'))
        write_json(tmp/'encoder.json',dict(contract=contract,backbone_sha=BACKBONE_SHA,exit_spec=bank.spec,
            selected=dict(backbone=source_info,exits=SELECTED),depths=DEPTHS,dimensions=[WIDTH],max_length=MAX_LENGTH,
            pooling='masked_mean_including_special_tokens',full_depth_skips_exit_heads=True))
        shutil.copyfile(ROOT/'LICENSE_PROVENANCE.json',tmp/'LICENSE_PROVENANCE.json')
        if (SOURCE_RUN/'notices').exists():shutil.copytree(SOURCE_RUN/'notices',tmp/'notices')
        write_json(tmp/'COMMITTED.json',dict(contract=CONTRACT_ID,checksums={str(p.relative_to(tmp)):file_sha(p) for p in tmp.rglob('*') if p.is_file()}))
        EXPORT.parent.mkdir(exist_ok=True);os.replace(tmp,EXPORT)
    manifest=json.loads((EXPORT/'COMMITTED.json').read_text());assert manifest['contract']==CONTRACT_ID
    assert all(file_sha(EXPORT/p)==sha for p,sha in manifest['checksums'].items())
    probe=['forgot my password','The library lends books.'];before=encode_profiles(probe,bank)
    original_backbone,original_tokenizer=backbone,tokenizer
    try:
        backbone=AutoModel.from_pretrained(EXPORT/'backbone',attn_implementation='sdpa').to(DEVICE).eval().requires_grad_(False)
        tokenizer=AutoTokenizer.from_pretrained(EXPORT/'tokenizer')
        loaded=ExitBank().to(DEVICE);loaded.load_state_dict(load_file(str(EXPORT/'exits.safetensors')));loaded.eval()
        after=encode_profiles(probe,loaded)
        for d in before:torch.testing.assert_close(before[d],after[d],atol=1e-5,rtol=1e-4)
        del loaded
    finally:backbone,tokenizer=original_backbone,original_tokenizer
    print('Verified export/reload:',EXPORT)


## Final sanity and run orchestration
The function below checks every exit at 384D and verifies unchanged backbone weights.
The final cell runs both complete experiments sequentially, then shows a matched comparison.
Set `RELEASE_WHEN_DONE=True` in that cell to flush Drive before disconnecting after both runs.


In [ ]:
def check_final():
    global FINAL_SANITY, final_sanity
    @torch.inference_mode()
    def final_sanity():
        bank.eval();texts=[case[i] for i in range(3) for case in SANITY_CASES]
        vectors=encode_profiles(texts,bank)
        first=encode_profiles([SANITY_CASES[0][0]],bank)
        repeat=encode_profiles([SANITY_CASES[0][0]],bank)
        padded=encode_profiles([SANITY_CASES[0][0],'The library has books. '*40],bank)
        reports={}
        for d,raw in vectors.items():
            report=sanity_scores(raw,SANITY_CASES,[WIDTH])
            report['repeatable']=torch.allclose(first[d],repeat[d],atol=1e-5,rtol=1e-4)
            report['padding_cosine']=float(F.cosine_similarity(first[d],padded[d][:1]).item())
            assert report['finite'] and report['nonzero'] and report['repeatable'] and report['padding_cosine']>.999
            reports[str(d)]=report
            print('depth',d,'384D',report['profiles'][str(WIDTH)]['hit_at_1'],'padding',report['padding_cosine'])
        assert backbone_digest()==BACKBONE_SHA
        write_json(ROOT/'evaluations'/'final-sanity.json',reports)
        return reports
    FINAL_SANITY=final_sanity()
    write_json(ROOT/'RUN_COMPLETE.json',dict(contract=CONTRACT_ID,selected=SELECTED,export=str(EXPORT),sanity_passed=True))


In [ ]:
RELEASE_WHEN_DONE=False
COMPARISON=[]
for SOURCE_MODE in VARIANTS:
    ROOT=EXPERIMENT_ROOT/SOURCE_MODE;LOCAL=EXPERIMENT_LOCAL/SOURCE_MODE
    ROOT.mkdir(parents=True,exist_ok=True);LOCAL.mkdir(parents=True,exist_ok=True)
    seed_everything(SEED)
    print('\n===',SOURCE_MODE,'===',flush=True)
    load_variant()
    check_execution()
    prepare_data()
    train_heads()
    evaluate_retrieval()
    export_variant()
    check_final()
    COMPARISON.extend(dict(variant=SOURCE_MODE,**r) for r in RESULTS)
    # Free the first GPU model and optimizer before loading the second.
    del bank,backbone,optimizer,scheduler,sampler
    gc.collect();torch.cuda.empty_cache()
write_json(EXPERIMENT_ROOT/'comparison.json',COMPARISON)
import pandas as pd
frame=pd.DataFrame(COMPARISON)
frame.to_csv(EXPERIMENT_ROOT/'comparison.csv',index=False)
display(frame.pivot_table(index=['dataset','depth','mode'],columns='variant',values='ndcg@10'))
print('Both variants completed; exports and final sanity reports:',EXPERIMENT_ROOT)
if RELEASE_WHEN_DONE:
    from google.colab import runtime
    for marker in EXPERIMENT_ROOT.rglob('COMMITTED.json'):
        if any(p.startswith('.uploading-') for p in marker.relative_to(EXPERIMENT_ROOT).parts):continue
        meta=json.loads(marker.read_text())
        files=meta['checksums'] if 'checksums' in meta else {'state.pt':meta['sha256']}
        assert all(file_sha(marker.parent/p)==sha for p,sha in files.items())
    assert os.path.ismount('/content/drive'),'Drive must be mounted before flushing.'
    drive.flush_and_unmount(timeout_ms=20*60*1000)
    runtime.unassign()
